# Multiperiod Heat Pumps

**Learning outcome:** Apply multiperiod heat pumps through the public `PinchProblem` or `PinchWorkspace` workflow.

**Level:** Advanced  
**Execution profile:** `slow-hpr`  
**Expected runtime:** 5 to 30 minutes  
**Optional extras:** hpr

The lifecycle is explicit: prepare the study, run the named method, then inspect cached results. Observation cells do not launch analysis.

## Study question and data

**Study question:** Does one heat-pump concept remain useful across all operating periods?

The sample data is packaged with OpenPinch, so the notebook runs without path setup. Read the named inputs and assumptions before substituting plant data.

## Step 1: Run period Carnot screening

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [ ]:
from OpenPinch import PinchProblem

problem = PinchProblem("crude_preheat_train_multiperiod.json", project_name="Crude HPR")
def screen_periods(method, **arguments):
    try:
        return {"status": "feasible", "results": method(**arguments)}
    except ValueError as error:
        return {"status": "no shared feasible solution", "reason": str(error)}

period_heat_pumps = problem.target.all_periods.carnot_heat_pump(
    load_fraction=0.25, maximum_restarts=1
)
weighted = problem.summary_frame(include_weighted_average=True)
weighted

## Step 2: Compare simulated and MVR period results

Run this cell once, then inspect its named outputs. Arguments on the method call apply to this analysis; stored configuration is only the fallback when an argument is omitted.

In [ ]:
period_refrigeration = screen_periods(
    problem.target.all_periods.carnot_refrigeration,
    load_fraction=0.25, maximum_restarts=1
)
period_vc_heat_pumps = screen_periods(
    problem.target.all_periods.vapour_compression_heat_pump,
    refrigerants=["water"], load_fraction=0.25,
    maximum_restarts=1,
)
period_vc_refrigeration = screen_periods(
    problem.target.all_periods.vapour_compression_refrigeration,
    refrigerants=["ammonia"], load_fraction=0.25,
    maximum_restarts=1,
)
period_mvr = screen_periods(
    problem.target.all_periods.mvr_heat_pump,
    load_fraction=0.25, maximum_restarts=1
)
list(problem.period_results)

## Review the result

Use the weighted summary and period-screen outcomes to identify which operating condition controls capacity, lift, and feasibility.

In [ ]:
from IPython.display import display

period_screen = {
    "Carnot heat pump": period_heat_pumps,
    "Carnot refrigeration": period_refrigeration,
    "vapour compression heat pump": period_vc_heat_pumps,
    "vapour compression refrigeration": period_vc_refrigeration,
    "MVR": period_mvr,
}
display(weighted)
display(period_screen)

## Interpret the result

Inspect period feasibility and load before annual weighting; a shared concept must work at the operating extremes.

## Adapt this template

Use period-specific loads when plant availability or heat-source duty changes materially.

Keep the workflow explicit: prepare input, call one named engineering method, inspect cached results, then export.